# ETL Lesson 2 Practice 3 - Solución completa

Este notebook implementa los 4 paradigmas de extracción solicitados:
1. Full Extraction (Snapshot)
2. Incremental Extraction (Watermark)
3. Change Data Capture (CDC)
4. Streaming (Event-Based)

También genera archivos CSV y una comparación final con visualización.

## 1) Setup: imports y configuración

In [ ]:
import sqlite3
import json
import time
import random
import queue
import threading
from pathlib import Path
from datetime import datetime, timedelta, timezone

import pandas as pd
import requests
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

BASE_DIR = Path('.')
DB_PATH = BASE_DIR / 'production_source.db'
RAW_JSON_PATH = BASE_DIR / 'raw_users.json'

FULL_CSV = BASE_DIR / 'users_full.csv'
INCR_CSV = BASE_DIR / 'users_incremental.csv'
CDC_CSV = BASE_DIR / 'users_cdc.csv'
STREAM_CSV = BASE_DIR / 'users_streaming.csv'

print('Working directory:', BASE_DIR.resolve())
print('Database path:', DB_PATH.resolve())

## 2) Fase 1 - Adquisición desde API y aplanamiento

In [ ]:
API_URL = 'https://jsonplaceholder.typicode.com/users'

response = requests.get(API_URL, timeout=30)
response.raise_for_status()
raw_users = response.json()

with open(RAW_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(raw_users, f, ensure_ascii=False, indent=2)

print(f'Usuarios recuperados: {len(raw_users)}')
print(f'Raw JSON guardado en: {RAW_JSON_PATH}')
raw_users[0]

In [ ]:
# Flatten: dejamos una estructura tabular limpia
flattened_users = []
for user in raw_users:
    flattened_users.append({
        'id': int(user['id']),
        'name': str(user['name']),
        'email': str(user['email']),
        'city': str(user['address']['city']),
        'company': str(user['company']['name'])
    })

df_flat = pd.DataFrame(flattened_users).sort_values('id').reset_index(drop=True)
print(df_flat.shape)
df_flat.head()

## 3) Fase 1 - Construcción del Source DB (SQLite)

In [ ]:
# Reiniciar la base para que el notebook sea idempotente
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute('''
CREATE TABLE users (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL,
    city TEXT NOT NULL,
    company TEXT NOT NULL,
    updated_at TEXT NOT NULL
)
''')

cur.execute('''
CREATE TABLE cdc_log (
    event_id INTEGER PRIMARY KEY AUTOINCREMENT,
    op TEXT NOT NULL,
    user_id INTEGER,
    name TEXT,
    email TEXT,
    city TEXT,
    company TEXT,
    event_time TEXT NOT NULL
)
''')

# Cargar estado inicial con timestamp de "ayer"
yesterday = datetime.now(timezone.utc) - timedelta(days=1)
yesterday_str = yesterday.isoformat()

records = [
    (u['id'], u['name'], u['email'], u['city'], u['company'], yesterday_str)
    for u in flattened_users
]

cur.executemany(
    'INSERT INTO users (id, name, email, city, company, updated_at) VALUES (?, ?, ?, ?, ?, ?)',
    records
)
conn.commit()

print('Filas iniciales en users:', cur.execute('SELECT COUNT(*) FROM users').fetchone()[0])
print('Watermark inicial (ayer):', yesterday_str)

## 4) Fase 1 - Simulación de actividad (UPDATE, INSERT, DELETE)

In [ ]:
now = datetime.now(timezone.utc)
now_str = now.isoformat()

# 1) UPDATE user #1
new_email_user1 = 'sincere.april@biz.com'
cur.execute(
    'UPDATE users SET email = ?, updated_at = ? WHERE id = 1',
    (new_email_user1, now_str)
)
cur.execute('SELECT id, name, email, city, company FROM users WHERE id = 1')
u1 = cur.fetchone()
cur.execute(
    'INSERT INTO cdc_log (op, user_id, name, email, city, company, event_time) VALUES (?, ?, ?, ?, ?, ?, ?)',
    ('UPDATE', *u1, now_str)
)

# 2) INSERT user #99
new_user = (99, 'Data Engineer', 'de99@example.com', 'Madrid', 'ETL Labs', now_str)
cur.execute(
    'INSERT INTO users (id, name, email, city, company, updated_at) VALUES (?, ?, ?, ?, ?, ?)',
    new_user
)
cur.execute(
    'INSERT INTO cdc_log (op, user_id, name, email, city, company, event_time) VALUES (?, ?, ?, ?, ?, ?, ?)',
    ('INSERT', 99, 'Data Engineer', 'de99@example.com', 'Madrid', 'ETL Labs', now_str)
)

# 3) DELETE user #2 (y log CDC antes de borrar)
cur.execute('SELECT id, name, email, city, company FROM users WHERE id = 2')
u2 = cur.fetchone()
if u2:
    cur.execute(
        'INSERT INTO cdc_log (op, user_id, name, email, city, company, event_time) VALUES (?, ?, ?, ?, ?, ?, ?)',
        ('DELETE', *u2, now_str)
    )
cur.execute('DELETE FROM users WHERE id = 2')

conn.commit()

print('Simulación completada.')
print('Filas actuales en users:', cur.execute('SELECT COUNT(*) FROM users').fetchone()[0])
print('Eventos en cdc_log:', cur.execute('SELECT COUNT(*) FROM cdc_log').fetchone()[0])

## 5) Fase 2 - Método A: Full Extraction

In [ ]:
df_full = pd.read_sql_query('SELECT * FROM users ORDER BY id', conn)
df_full.to_csv(FULL_CSV, index=False)

print('Rows Full:', len(df_full))
print('CSV:', FULL_CSV)
df_full.head()

## 6) Fase 2 - Método B: Incremental (Watermark)

In [ ]:
last_watermark = yesterday_str
query_incremental = '''
SELECT *
FROM users
WHERE updated_at > ?
ORDER BY id
'''

df_incremental = pd.read_sql_query(query_incremental, conn, params=(last_watermark,))
df_incremental.to_csv(INCR_CSV, index=False)

print('Rows Incremental:', len(df_incremental))
print('CSV:', INCR_CSV)
df_incremental

## 7) Fase 2 - Método C: CDC

In [ ]:
query_cdc = '''
SELECT op, user_id, name, email, city, company, event_time
FROM cdc_log
WHERE event_time > ?
ORDER BY event_id
'''

df_cdc = pd.read_sql_query(query_cdc, conn, params=(last_watermark,))
df_cdc.to_csv(CDC_CSV, index=False)

print('Rows CDC:', len(df_cdc))
print('CSV:', CDC_CSV)
df_cdc

## 8) Fase 2 - Método D: Streaming (simulado con Queue)

In [ ]:
event_queue = queue.Queue()
stream_events = []

EVENT_TYPES = ['USER_LOGIN', 'PAGE_VIEW', 'ADD_TO_CART', 'LOGOUT', 'CLICK_BUTTON']


def producer(num_events=12, min_delay=0.05, max_delay=0.2):
    for i in range(num_events):
        evt = {
            'event_id': i + 1,
            'event_type': random.choice(EVENT_TYPES),
            'user_id': random.choice([1, 3, 4, 5, 99]),
            'event_time': datetime.now(timezone.utc).isoformat()
        }
        event_queue.put(evt)
        time.sleep(random.uniform(min_delay, max_delay))
    event_queue.put(None)  # señal de fin


def consumer():
    while True:
        msg = event_queue.get()
        if msg is None:
            break
        stream_events.append(msg)

p = threading.Thread(target=producer)
c = threading.Thread(target=consumer)

p.start()
c.start()

p.join()
c.join()

df_stream = pd.DataFrame(stream_events)
df_stream.to_csv(STREAM_CSV, index=False)

print('Rows Streaming:', len(df_stream))
print('CSV:', STREAM_CSV)
df_stream.head()

## 9) Fase 3 - Comparación y análisis

In [ ]:
counts = pd.DataFrame([
    {'method': 'Full Extraction', 'rows': len(df_full)},
    {'method': 'Incremental', 'rows': len(df_incremental)},
    {'method': 'CDC', 'rows': len(df_cdc)},
    {'method': 'Streaming', 'rows': len(df_stream)}
])

counts

In [ ]:
ax = counts.plot(kind='bar', x='method', y='rows', legend=False, figsize=(10, 5), color=['#4c78a8', '#f58518', '#54a24b', '#e45756'])
ax.set_title('Registros procesados por método de extracción')
ax.set_xlabel('Método')
ax.set_ylabel('Número de registros')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Respuestas automáticas solicitadas por la práctica
max_method = counts.loc[counts['rows'].idxmax(), 'method']

delete_in_full = (df_full['id'] == 2).any()
delete_in_incremental = (df_incremental['id'] == 2).any() if 'id' in df_incremental.columns else False
delete_in_cdc = ((df_cdc['op'] == 'DELETE') & (df_cdc['user_id'] == 2)).any()

print('1) Volume - método con más filas:', max_method)
print('2) Completeness - ¿quién falla en detectar delete?')
print('   - Full Extraction detecta delete explícito?:', delete_in_full)
print('   - Incremental detecta delete explícito?:', delete_in_incremental)
print('   - CDC detecta delete explícito?:', delete_in_cdc)
print('3) Latency - mejor para <=5s:', 'Streaming (event-based)')

In [ ]:
# Cerrar conexión
conn.close()
print('Notebook finalizado y conexión cerrada.')